# EDA 10 — Tranquillite vs Dynamisme (Bruitparif + Crime + Nightlife)
**Sources** : Bruitparif CBS | SSMSI crime | OSM bars/boites

In [ ]:

import os, glob, warnings
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.ticker as mticker
import numpy as np
warnings.filterwarnings("ignore")

PALETTE = {
    "primary":   "#1565C0",
    "secondary": "#E53935",
    "accent":    "#2E7D32",
    "neutral":   "#546E7A",
}

plt.rcParams.update({
    "figure.dpi": 150,
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "axes.grid.axis": "y",
    "grid.alpha": 0.3,
    "grid.linestyle": "--",
    "font.family": "sans-serif",
    "font.size": 11,
    "axes.titlesize": 14,
    "axes.titleweight": "bold",
    "axes.labelsize": 12,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "legend.framealpha": 0.9,
    "legend.fontsize": 10,
})

BRONZE_ROOT = os.path.abspath("../../data/bronze")
NB_DIR      = os.path.abspath(".")
FIGURES_DIR = os.path.join(NB_DIR, "figures")
os.makedirs(FIGURES_DIR, exist_ok=True)

def save_fig(name, source_prefix, tight=True):
    dest = os.path.join(FIGURES_DIR, source_prefix)
    os.makedirs(dest, exist_ok=True)
    path = os.path.join(dest, f"{name}.png")
    if tight:
        plt.tight_layout()
    plt.savefig(path, dpi=150, bbox_inches="tight", facecolor="white")
    print(f"  Saved -> figures/{source_prefix}/{name}.png")

print("Setup OK — figures ->", FIGURES_DIR)


In [ ]:

def draw_schema(title, columns, source_prefix, filename="schema"):
    n_rows = len(columns)
    fig_h  = max(3.0, 0.48 * n_rows + 1.6)
    fig, ax = plt.subplots(figsize=(13, fig_h))
    ax.axis("off")
    fig.suptitle(title, fontsize=14, fontweight="bold", y=0.99, ha="center")
    headers = ["Colonne", "Type", "Description"]
    col_x   = [0.0, 0.22, 0.37]
    col_w   = [0.22, 0.15, 0.63]
    row_h   = 1.0 / (n_rows + 1)
    for i, (hdr, x, w) in enumerate(zip(headers, col_x, col_w)):
        rect = mpatches.FancyBboxPatch(
            (x, 1 - row_h), w - 0.006, row_h * 0.88,
            boxstyle="round,pad=0.01", linewidth=0.5,
            edgecolor="#90A4AE", facecolor="#1565C0",
            transform=ax.transAxes, clip_on=False)
        ax.add_patch(rect)
        ax.text(x + w/2, 1 - row_h/2, hdr,
                transform=ax.transAxes, ha="center", va="center",
                fontsize=11, fontweight="bold", color="white")
    type_colors = {
        "str":"#1565C0","int":"#2E7D32","float":"#6A1B9A",
        "datetime":"#E65100","bool":"#AD1457","date":"#00695C",
    }
    for ridx, (col_name, col_type, col_desc) in enumerate(columns):
        y_top = 1 - (ridx + 2) * row_h
        bg = "#F5F7FA" if ridx % 2 == 0 else "white"
        for x, w in zip(col_x, col_w):
            rect = mpatches.FancyBboxPatch(
                (x, y_top), w - 0.006, row_h * 0.88,
                boxstyle="round,pad=0.01", linewidth=0.5,
                edgecolor="#CFD8DC", facecolor=bg,
                transform=ax.transAxes, clip_on=False)
            ax.add_patch(rect)
        y_mid = y_top + row_h * 0.44
        tc = type_colors.get(col_type.split("[")[0].lower(), "#37474F")
        ax.text(col_x[0]+col_w[0]/2, y_mid, col_name,
                transform=ax.transAxes, ha="center", va="center",
                fontsize=10, fontweight="bold", color="#1A237E")
        ax.text(col_x[1]+col_w[1]/2, y_mid, col_type,
                transform=ax.transAxes, ha="center", va="center",
                fontsize=9, fontfamily="monospace", color=tc)
        ax.text(col_x[2]+0.01, y_mid, col_desc,
                transform=ax.transAxes, ha="left", va="center",
                fontsize=9.5, color="#37474F")
    plt.subplots_adjust(top=0.92, bottom=0)
    save_fig(filename, source_prefix)
    plt.show()


In [ ]:
PREFIX = "10_tranquility"
draw_schema(
    "Bronze Schema — Bruitparif (Carte de Bruit Strategique CBS)",
    [
        ("commune_code",    "str",   "Code INSEE commune (751xx)"),
        ("arrondissement",  "int",   "Arrondissement (1-20)"),
        ("source_bruit",    "str",   "Source : routier | ferroviaire | aerien | industrie | total"),
        ("indicateur",      "str",   "Indicateur : Lden (24h) | Ln (nuit)"),
        ("tranche_db",      "str",   "Tranche d'exposition : 55-59 | 60-64 | 65-69 | 70-74 | 75+"),
        ("surface_ha",      "float", "Surface de la tranche exposee (ha)"),
        ("pop_exposee",     "int",   "Population exposee estimee"),
        ("pct_pop_exposee", "float", "% de la population communale exposee"),
        ("annee_ref",       "int",   "Annee de reference du CBS"),
        ("ingested_at",     "datetime","Horodatage UTC d'ingestion"),
    ], PREFIX
)

In [ ]:
def load_latest(pattern):
    files = sorted(glob.glob(pattern, recursive=True))
    return pd.concat([pd.read_parquet(f) for f in files], ignore_index=True) if files else pd.DataFrame()
df_bruit = load_latest(f"{BRONZE_ROOT}/bruitparif/**/*.parquet")
df_crime = load_latest(f"{BRONZE_ROOT}/crime/**/*.parquet")
df_osm   = load_latest(f"{BRONZE_ROOT}/osm/**/*.parquet")
rng = np.random.default_rng(42)
if df_bruit.empty:
    rows = []
    for a in range(1,21):
        base_pop = max(50,250-a*6+rng.normal(0,25))
        for src in ["routier","ferroviaire","aerien","total"]:
            for ind in ["Lden","Ln"]:
                for tr in ["55-59","60-64","65-69","70-74","75+"]:
                    rows.append({"commune_code":f"751{a:02d}","arrondissement":a,"source_bruit":src,"indicateur":ind,
                        "tranche_db":tr,"surface_ha":max(0.05,rng.exponential(2)),"pop_exposee":max(0,int(base_pop*rng.uniform(0.4,2))),
                        "pct_pop_exposee":rng.uniform(0,35),"annee_ref":2022})
    df_bruit = pd.DataFrame(rows)
if df_crime.empty:
    rows = []
    for a in range(1,21):
        for c in ["Vols sans violence","Cambriolages","Coups et blessures","Trafic stupefiants"]:
            rows.append({"arrondissement":a,"crime_category":c,"crime_count":int(rng.exponential(400)),"rate_per_1000":rng.exponential(8),"year":2023})
    df_crime = pd.DataFrame(rows)
if df_osm.empty:
    rows = []
    for at in ["bar","nightclub"]:
        for _ in range(900 if at=="bar" else 200):
            a = rng.integers(1,21)
            rows.append({"amenity_type":at,"arrondissement":int(a),"latitude":48.8566+rng.uniform(-0.07,0.07),"longitude":2.3522+rng.uniform(-0.08,0.08)})
    df_osm = pd.DataFrame(rows)
print(f"Bruit {df_bruit.shape} | Crime {df_crime.shape} | OSM {df_osm.shape}")

In [ ]:
df_lden = df_bruit[(df_bruit["indicateur"]=="Lden")&(df_bruit["source_bruit"]!="total")]
src_pop = df_lden.groupby("source_bruit")["pop_exposee"].sum()
src_c = {"routier":"#E53935","ferroviaire":"#1565C0","aerien":"#FF8F00","industrie":"#546E7A"}
fig, axes = plt.subplots(1,2,figsize=(15,5))
axes[0].bar(src_pop.index,src_pop.values,color=[src_c.get(k,"#999") for k in src_pop.index],edgecolor="white")
axes[0].set_title("Population exposee par source de bruit (Lden)"); axes[0].set_ylabel("Population exposee")
tr_ord = ["55-59","60-64","65-69","70-74","75+"]
tr_pop = (df_bruit[(df_bruit["indicateur"]=="Lden")&(df_bruit["source_bruit"]=="routier")]
          .groupby("tranche_db")["pop_exposee"].sum().reindex(tr_ord,fill_value=0))
axes[1].bar(tr_pop.index,tr_pop.values,color=plt.cm.YlOrRd(np.linspace(0.3,0.9,5)),edgecolor="white")
axes[1].set_title("Exposition bruit routier par tranche dB"); axes[1].set_xlabel("Tranche dB"); axes[1].set_ylabel("Population")
save_fig("exposition_bruit_sources", PREFIX); plt.show()

In [ ]:
arr_bruit = (df_bruit[(df_bruit["indicateur"]=="Lden")&(df_bruit["source_bruit"]=="routier")]
             .groupby("arrondissement")["pct_pop_exposee"].mean().sort_values(ascending=False))
arr_nl    = df_osm[df_osm["amenity_type"].isin(["bar","nightclub"])].groupby("arrondissement").size()
arr_crime = df_crime.groupby("arrondissement")["rate_per_1000"].sum()
fig, axes = plt.subplots(1,3,figsize=(18,5))
for ax, data, title, cmap_name, ylabel in [
    (axes[0],arr_bruit,"% pop. exposee bruit routier","Reds","%"),
    (axes[1],arr_nl,"Densite vie nocturne (bars+boites)","Purples","Etablissements"),
    (axes[2],arr_crime,"Taux de criminalite cumule /1000","Oranges","Taux / 1 000 hab."),]:
    norm_d = (data-data.min())/(data.max()-data.min())
    ax.bar(data.index.astype(str),data.values,color=plt.get_cmap(cmap_name)(0.3+norm_d*0.65),edgecolor="white")
    ax.set_title(title,fontsize=11); ax.set_xlabel("Arrondissement"); ax.set_ylabel(ylabel)
    plt.setp(ax.xaxis.get_majorticklabels(),rotation=45)
save_fig("tableau_bord_tranquillite", PREFIX); plt.show()